# 06 GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
    plot_latest_si_vs_future_si,
    summarize_pathogen_coverage,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Parameters

In [43]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)
FIGURE_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.figure_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = cfg_path(PIPELINE_CONFIG, "paths.assets_dir", PROJECT_ROOT)

ANTIBIOTIC_ANALYSIS_LEVEL = cfg(PIPELINE_CONFIG, "glass_interactive.antibiotic_analysis_level")
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]

GLASS_XLSX_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_xlsx", PROJECT_ROOT)
GLASS_CLEANED_PATH = cfg_path(PIPELINE_CONFIG, "paths.glass_cleaned", PROJECT_ROOT)
BACTERIA_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.pathogens", PROJECT_ROOT)
ANTIBIOTIC_SYNONYM_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_synonyms", PROJECT_ROOT)
ANTIBIOTIC_COARSENING_PATH = cfg_path(PIPELINE_CONFIG, "paths.antibiotic_coarsening", PROJECT_ROOT)

# Reuse SI fit window settings so interactive plots stay aligned with notebook 05 outputs.
FIT_START_YEAR = cfg(PIPELINE_CONFIG, "si.fit_start_year")
FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.fit_end_year")
CURVE_FIT_END_YEAR = cfg(PIPELINE_CONFIG, "si.curve_fit_end_year")
NUM_FUTURE_YEARS = cfg(PIPELINE_CONFIG, "glass_interactive.num_future_years")
LOESS_FRAC = cfg(PIPELINE_CONFIG, "si.loess_frac")
LOESS_DEGREE = cfg(PIPELINE_CONFIG, "si.loess_degree")
TAIL_DAMPING_YEARS = cfg(PIPELINE_CONFIG, "si.tail_damping_years")

# Interactive filtering and plot layout controls.
MIN_LAST_AVAILABLE_COUNT = cfg(PIPELINE_CONFIG, "glass_interactive.min_last_available_count")
MIN_TOTAL_PUBLICATIONS = cfg(PIPELINE_CONFIG, "glass_interactive.min_total_publications")
MIN_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.min_count_abs_pct_error_mean")
MAX_COUNT_ABS_PCT_ERROR_MEAN = cfg(PIPELINE_CONFIG, "glass_interactive.max_count_abs_pct_error_mean")
MIN_COUNTS_BACKGROUND = cfg(PIPELINE_CONFIG, "glass_interactive.min_counts_background")
MIN_ABS_DELTA_SI = cfg(PIPELINE_CONFIG, "glass_interactive.min_abs_delta_si")
MIN_SI_DELTA = cfg(PIPELINE_CONFIG, "glass_interactive.min_si_delta")
DISPLAY_NAME = cfg(PIPELINE_CONFIG, "glass_interactive.display_name")
MAX_TEXT_LABELS = cfg(PIPELINE_CONFIG, "glass_interactive.max_text_labels")
WIDTH = cfg(PIPELINE_CONFIG, "glass_interactive.width")
HEIGHT = cfg(PIPELINE_CONFIG, "glass_interactive.height")

if not YEARLY_SCREENED_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SCREENED_PATH}. Run notebook 04 to create screened AMR pairs first.")
if not YEARLY_SI_PATH.exists():
    raise FileNotFoundError(f"Missing {YEARLY_SI_PATH}. Run notebook 05 after notebook 04 to build SI outputs.")
if YEARLY_SI_PATH.stat().st_mtime < YEARLY_SCREENED_PATH.stat().st_mtime:
    raise RuntimeError(
        f"{YEARLY_SI_PATH} is older than {YEARLY_SCREENED_PATH}. "
        "Run notebook 05 after the latest notebook 04 LLM/screened-pair run."
    )


## Load SI And Clean GLASS List

In [44]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)

si_source_df = pd.read_parquet(YEARLY_SCREENED_PATH)
si_pathogen_summary_df = summarize_pathogen_coverage(
    source_df=si_source_df,
    modeled_df=si_df,
    source_label="screened AMR input from notebook 04",
    modeled_label="filtered SI dataset from notebook 05",
)
display(si_pathogen_summary_df.query("status != 'present_in_both'"))
display(glass_pairs_df.head(20))

Loaded SI rows: 6,601 from /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si.parquet
Pathogens: 27; antibiotics/levels: 65
Saved cleaned GLASS rows: 89 to /Users/f.costa/Code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv
screened AMR input from notebook 04 pathogens: 48
filtered SI dataset from notebook 05 pathogens: 27
Pathogens present in screened AMR input from notebook 04 but absent from filtered SI dataset from notebook 05: 21
Aggregatibacter actinomycetemcomitans, Bacillus anthracis, Bordetella pertussis, Borrelia burgdorferi, Brucella melitensis, Burkholderia cepacia, Burkholderia pseudomallei, Chlamydia pneumoniae, Chlamydia trachomatis, Coxiella burnetii, Francisella tularensis, Legionella pneumophila, Moraxella catarrhalis, Mycobacterium avium, Mycobacterium leprae, Mycoplasmoides pneumoniae, Neisseria meningitidis, Orientia tsutsugamushi, Porphyromonas gingivalis, Treponema pallidum, Yersinia pestis


,pathogen,source_rows,source_publications,source_pairs,modeled_rows,modeled_publications,modeled_pairs,in_source,in_modeled,status
0,Aggregatibacter actinomycetemcomitans,77,26,19,0.0,0.0,0.0,True,False,missing_from_modeled
1,Bacillus anthracis,47,31,31,0.0,0.0,0.0,True,False,missing_from_modeled
2,Bordetella pertussis,65,62,23,0.0,0.0,0.0,True,False,missing_from_modeled
3,Borrelia burgdorferi,24,14,16,0.0,0.0,0.0,True,False,missing_from_modeled
4,Brucella melitensis,112,83,30,0.0,0.0,0.0,True,False,missing_from_modeled
5,Burkholderia cepacia,182,97,42,0.0,0.0,0.0,True,False,missing_from_modeled
6,Burkholderia pseudomallei,131,95,35,0.0,0.0,0.0,True,False,missing_from_modeled
7,Chlamydia pneumoniae,29,8,20,0.0,0.0,0.0,True,False,missing_from_modeled
8,Chlamydia trachomatis,90,28,31,0.0,0.0,0.0,True,False,missing_from_modeled
9,Coxiella burnetii,7,3,4,0.0,0.0,0.0,True,False,missing_from_modeled


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter baumannii,Amikacin,True,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter baumannii,Colistin,True,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter baumannii,Doripenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter baumannii,Gentamicin,True,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter baumannii,Imipenem/cilastatin,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter baumannii,Meropenem,True,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter baumannii,Minocycline,True,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter baumannii,Tigecycline,True,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [45]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    source_path=YEARLY_SI_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    curve_fit_end_year=CURVE_FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    verbose=True,
)
display(rank_df)

Loaded cached rank rows: 279 from /Users/f.costa/Code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future.parquet


,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,curve_fit_end_year,post_fit_validation_end_year,raw_latest_available_year,...,tail_damping_years,validation_loess_frac,validation_loess_degree,validation_tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Staphylococcus aureus,Azithromycin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,104.0,29.0,75.0,75.0
1,Clostridioides difficile,Moxifloxacin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,73.0,139.0,-66.0,66.0
2,Klebsiella pneumoniae,Amoxicillin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,190.0,138.0,52.0,52.0
3,Escherichia coli,Sulbactam,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,215.0,165.0,50.0,50.0
4,Escherichia coli,Ampicillin/sulbactam,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,153.0,104.0,49.0,49.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,Salmonella enterica,Ciprofloxacin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,18.0,18.0,0.0,0.0
275,Pseudomonas aeruginosa,Levofloxacin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,208.0,208.0,0.0,0.0
276,Salmonella enterica,Spectinomycin,2025,2028,3,2000,2023,2025,2025,2025,...,None,0.6,1,None,ok,None,272.0,272.0,0.0,0.0
277,Streptococcus pneumoniae,Ciprofloxacin,2025,2028,3,2000,2023,2025,2025,2026,...,None,0.6,1,None,ok,None,278.0,278.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [46]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_total_publications=MIN_TOTAL_PUBLICATIONS,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "interactive_latest_observed_count_vs_future_si_shift.pdf",
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")
if latest_count_shift_result["pdf_path"]:
    print(f"Saved PDF: {latest_count_shift_result['pdf_path']}")
elif latest_count_shift_result.get("pdf_error"):
    print(f"PDF export failed: {latest_count_shift_result['pdf_error']}")

Displayed rows: 86; GLASS matches: 31
Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/interactive_latest_observed_count_vs_future_si_shift.pdf


## Display GLASS Matches In Plot

In [47]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Klebsiella pneumoniae,Colistin,2025,2028,58.0,378.0,100.303120,0.668525,0.817665,0.149140,8.782740,antibiotic
1,Escherichia coli,Ciprofloxacin,2025,2028,52.0,758.0,53.101508,0.485264,0.480479,-0.004784,11.730286,antibiotic
2,Klebsiella pneumoniae,Meropenem,2025,2028,48.0,325.0,91.885064,0.499676,0.630865,0.131189,7.151396,antibiotic
3,Staphylococcus aureus,Oxacillin,2025,2028,44.0,550.0,56.488618,0.220244,0.265945,0.045701,22.710845,antibiotic
4,Acinetobacter baumannii,Colistin,2025,2028,44.0,354.0,45.798271,0.407430,0.438360,0.030930,13.383112,antibiotic
5,Escherichia coli,Meropenem,2025,2028,41.0,301.0,59.124157,0.201059,0.286774,0.085715,39.499782,antibiotic
6,Klebsiella pneumoniae,Ceftazidime,2025,2028,36.0,227.0,77.309769,0.606749,0.798238,0.191489,2.785433,antibiotic
7,Pseudomonas aeruginosa,Meropenem,2025,2028,36.0,339.0,45.998735,0.138025,0.168527,0.030502,4.867275,antibiotic
8,Pseudomonas aeruginosa,Colistin,2025,2028,36.0,220.0,45.220433,0.157103,0.273376,0.116273,33.843980,antibiotic
9,Acinetobacter baumannii,Meropenem,2025,2028,34.0,220.0,32.777717,0.132460,0.180424,0.047964,55.968570,antibiotic


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [48]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 55


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,total_publications,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Klebsiella pneumoniae,Ampicillin,2025,2028,19.0,117.0,27.338625,0.166911,0.294847,0.127936,15.000425
1,Staphylococcus aureus,Cefoxitin,2025,2028,37.0,238.0,49.699241,0.695541,0.814822,0.119281,10.806106
2,Escherichia coli,Tigecycline,2025,2028,19.0,115.0,22.746370,0.153973,0.270548,0.116575,19.270384
3,Mycobacterium tuberculosis,Rifampicin,2025,2028,22.0,172.0,27.107392,0.565306,0.673245,0.107939,20.008999
4,Enterococcus faecium,Linezolid,2025,2028,20.0,121.0,24.253955,0.325554,0.431813,0.106258,23.554061
5,Staphylococcus aureus,Meticillin,2025,2028,470.0,3843.0,564.190322,0.559850,0.656940,0.097090,10.405435
6,Staphylococcus aureus,Ampicillin,2025,2028,36.0,269.0,47.065260,0.300809,0.397790,0.096981,16.088224
7,Enterococcus faecalis,Ampicillin,2025,2028,13.0,107.0,15.982896,0.170560,0.263073,0.092513,38.337843
8,Escherichia coli,Fosfomycin,2025,2028,21.0,147.0,22.096168,0.303693,0.393894,0.090201,33.111795
9,Escherichia coli,Amoxicillin/clavulanic-acid,2025,2028,20.0,136.0,18.508094,0.310403,0.397004,0.086602,40.983159


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [49]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    pdf_path=FIGURE_OUTPUT_DIR / "future_si_vs_si_shift.pdf",
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]
if future_si_shift_result["pdf_path"]:
    print(f"Saved PDF: {future_si_shift_result['pdf_path']}")
elif future_si_shift_result.get("pdf_error"):
    print(f"PDF export failed: {future_si_shift_result['pdf_error']}")

Saved PDF: /Users/f.costa/Code/Antimicrobial-Resistance/outputs/figures/future_si_vs_si_shift.pdf


In [50]:
latest_count_shift_df.to_csv(
    FIGURE_OUTPUT_DIR / "latest_count_shift_df.csv",
    index=False,
)